In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [6]:
import importlib
import boxbox.cleaning

importlib.reload(boxbox.cleaning)

<module 'boxbox.cleaning' from 'c:\\Users\\enric\\Desktop\\Box_Box\\boxbox\\cleaning.py'>

In [8]:
from boxbox.loading import load_json

from boxbox.cleaning import (
    prepare_lap_data,
    remove_pit_laps,
    remove_neutralised_laps,
    add_fuel_correction,
)

from boxbox.analysis import (
    tyre_degradation_rate,
    compare_driver_compounds
)

In [9]:
race_data = load_json(
    "../data/japan_gp/session_laptimes.json"
)

session_info = load_json(
    "../data/japan_gp/session_info.json"
)

rcm = load_json(
    "../data/japan_gp/rcm.json"
)

drivers = load_json(
    "../data/japan_gp/drivers.json"
)

In [11]:
japan_sc_laps = [
    22,
    23,
    24,
    25,
    26,
    27
]

laps = prepare_lap_data(df)

laps = remove_pit_laps(laps)

laps = remove_neutralised_laps(
    laps,
    japan_sc_laps
)

laps = add_fuel_correction(
    laps,
    total_laps=53
)

NameError: name 'df' is not defined

In [ ]:
ver = laps[
    laps["drv"] == "VER"
]

lec = laps[
    laps["drv"] == "LEC"
]

ant = laps[
    laps["drv"] == "ANT"
]

In [ ]:
degradation_data = laps[
    laps["life"] > 1
].copy()

In [ ]:
ver = degradation_data[
    degradation_data["drv"] == "VER"
]

lec = degradation_data[
    degradation_data["drv"] == "LEC"
]

ant = degradation_data[
    degradation_data["drv"] == "ANT"
]

In [ ]:
japan_results = pd.concat(
    [
        compare_driver_compounds("VER", ver),
        compare_driver_compounds("LEC", lec),
        compare_driver_compounds("ANT", ant)
    ],
    ignore_index=True
)

japan_results

,driver,compound,slope,r_squared,intercept
0,VER,MEDIUM,0.017953,0.103214,0.307618
1,VER,HARD,0.035381,0.250594,-0.004122
2,LEC,MEDIUM,0.060286,0.342756,0.101587
3,LEC,HARD,0.020178,0.185225,0.099437
4,ANT,MEDIUM,0.011874,0.019649,0.934180
5,ANT,HARD,0.056567,0.582882,-0.307730


In [ ]:
japan_degradation = pd.DataFrame([
    {
        "driver": "VER",
        **tyre_degradation_rate(ver, "MEDIUM")
    },
    {
        "driver": "VER",
        **tyre_degradation_rate(ver, "HARD")
    },
    {
        "driver": "LEC",
        **tyre_degradation_rate(lec, "MEDIUM")
    },
    {
        "driver": "LEC",
        **tyre_degradation_rate(lec, "HARD")
    },
    {
        "driver": "ANT",
        **tyre_degradation_rate(ant, "MEDIUM")
    },
    {
        "driver": "ANT",
        **tyre_degradation_rate(ant, "HARD")
    }
])

japan_degradation

,driver,compound,laps,slope,stderr,r_squared,intercept
0,VER,MEDIUM,20,0.017953,0.012473,0.103214,93.331429
1,VER,HARD,26,0.035381,0.012489,0.250594,92.761407
2,LEC,MEDIUM,15,0.060286,0.023153,0.342756,92.340794
3,LEC,HARD,29,0.020178,0.008145,0.185225,92.332569
4,ANT,MEDIUM,20,0.011874,0.019769,0.019649,92.522387
5,ANT,HARD,26,0.056567,0.009768,0.582882,91.249383


In [ ]:
japan_degradation["race"] = "Japan"

In [ ]:
japan_degradation = japan_degradation[
    [
        "race",
        "driver",
        "compound",
        "laps",
        "slope",
        "stderr",
        "r_squared"
    ]
]

In [ ]:
japan_degradation.to_csv(
    "../data/japan_degradation_results.csv",
    index=False
)